# 🔍 17 · Looking inside networks and segmenting structures voxel by voxel
### Class activation maps you can trust, encoder–decoders, multi-task learning and segmentation from image labels

**BiologyToResearch** · Part D — Deep learning on subtomograms (2017–2024) · Module 17 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/17_interpretability_segmentation.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 4–6 min to run everything); a GPU is used automatically if present |
| **Core frameworks** | Python 3, NumPy, SciPy, scikit-learn, PyTorch, Matplotlib |
| **Prerequisites** | Module 16 (3D CNNs, the 15 Å / 24³ subtomogram convention, `train_classifier`), Module 05 (PyTorch); Module 00 (mass-based contour level) |
| **Estimated time** | 150 min (reading + running + ✏️ exercises) |
| **Lab papers covered** | Zhao, Zhou *et al.* 2018 (Respond-CAM); Liu C *et al.* 2018 ICIP (SSN3D semantic segmentation); Liu C *et al.* 2018 BMVC (multi-task DSM-Net); Zhu *et al.* 2021 (CIVA-Net, weak supervision); Zeng *et al.* 2018 (EDSS3D weakly supervised coarse segmentation); Zhou *et al.* 2020 (COS-Net one-shot); Li R *et al.* 2019 (Faster-RCNN mitochondria); AttPNet and PUB-SalNet (cited only) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods, designed to teach the core idea of each
> paper on a laptop CPU in minutes. They do **not** reproduce the papers' architectures, datasets or numbers. **All
> molecules and data are generated by code** (nothing is downloaded): realistic stand-ins with the size and symmetry
> of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** class activation maps (CAM, Grad-CAM, Respond-CAM) and **implement** Grad-CAM and Respond-CAM for a
   3D CNN with PyTorch's automatic differentiation.
2. **Verify numerically** Respond-CAM's *sum-to-score* property — exact for a single fully connected layer (minus the
   bias), only approximate for deeper heads — and **explain** where the error comes from.
3. **Use** activation maps to **detect** a classifier that is right for the wrong reason (a shortcut).
4. **Build and train** a small 3D encoder–decoder that segments the central molecule of a crowded subtomogram, and
   **measure** Dice and IoU.
5. **Compare** a multi-task network (classification + segmentation + coarse structure recovery, loss weights 1:10:1)
   with single-task networks trained on the same small dataset.
6. **Build** a weak-supervision pipeline in the spirit of CIVA-Net: image-level labels → activation-map seeds +
   cross-image consensus → pseudo-masks → a segmentation network, and **quantify** what is lost.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | toolkit, templates, a crowded-subtomogram simulator with voxel masks | 15 min |
| 1 · Looking inside a classifier | Grad-CAM vs Respond-CAM; verify the sum-to-score property | 30 min |
| 2 · Right for the wrong reason | a gold-bead shortcut, exposed by activation maps | 15 min |
| 3 · Voxel-wise segmentation | a 3D encoder–decoder, Dice and IoU | 25 min |
| 4 · Multi-task learning | one encoder, three heads, weights 1:10:1 | 20 min |
| 5 · Segmentation from image labels | CAM seeds + consensus → pseudo-masks → train | 25 min |
| 6 · Other ways to find structures | one-shot segmentation, detection boxes, point sets, saliency | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Module 16 ended with classifiers that recognise six molecules from noisy subtomograms. Two questions remain before a
biologist should trust them. **Why** did the network say "proteasome"? A network can score well on a benchmark by
exploiting something irrelevant — a gold fiducial bead, the ice thickness of one tomogram — and then fail silently on
new cells. And **where** exactly is the molecule? Real subtomograms are crowded: neighbouring complexes, membranes and
filaments poke into every box, and they bias alignment and averaging (Module 13's "target segmentation" problem).
Outlining the target voxel by voxel — **semantic segmentation** — solves this, but drawing 3D masks by hand is
extremely slow. This notebook connects the two questions: maps that explain a classifier's decision can also become
cheap, approximate segmentation labels.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook is designed for a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from scipy.stats import kendalltau

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit
Copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0
print("🧰 toolkit loaded")

### 0.1 Crowded subtomograms with voxel-level ground truth

We keep Module 16's conventions — templates binned to **15 Å voxels**, **24³ boxes** (36 nm), a ±60° missing wedge,
white noise at **SNR 0.1** — and add **crowding**: every box holds a centred *target* (one of four classes: 70S
ribosome, 20S proteasome, capped proteasome, RuBisCO) plus 1–3 *neighbours* (any of seven zoo molecules) placed just
outside the target so that they poke into the box.

For segmentation we need a **ground-truth mask** of the target. We reuse Module 00's **mass-based contour**: a protein
of mass $M$ occupies about $V = M / (0.81\ \text{Da/Å}^3)$, so at 15 Å voxels the mask is the $n = V / 15^3$ brightest
voxels of the (rotated, noise-free) target density. *In words:* the mask has exactly the volume the molecule's mass
predicts. A 70S ribosome gets ≈ 830 voxels (6 % of the box); RuBisCO only ≈ 100 (0.7 %).

In [ ]:
# ── 0.1 Templates and the crowded-subtomogram simulator ──
VOXEL_A, BOX = 15.0, 24
TARGETS = ["4v4r", "3j9i", "6utj", "1bxn"]
NEIGHBOURS = ["4v4r", "3j9i", "6utj", "1bxn", "4cr2", "1qvr", "3cf3"]
SHORT = {"4v4r": "70S ribo", "3j9i": "20S prot", "6utj": "capped prot", "1bxn": "RuBisCO"}
NAMES = [SHORT[p] for p in TARGETS]
WEDGE = missing_wedge_mask((BOX,) * 3, 60)


def make_template(pid: str, box: int = BOX, voxel_A: float = VOXEL_A) -> np.ndarray:
    """Zoo map → box³ template at voxel_A Å/voxel (prepared at 10 Å, then resampled), scaled to [0, 1]."""
    raw, vs, src = zoo_map(pid)
    n10 = int(round(box * voxel_A / 10.0))
    v = ndimage.zoom(prepare_map(raw, size=n10, sigma=1.0), box / n10, order=1)
    v = np.clip(v, 0, None)
    print(f"  {pid.upper()} {ZOO[pid]['name']:26s} ← {src}")
    return (v / v.max()).astype(np.float32)


TEMPLATES = {pid: make_template(pid) for pid in NEIGHBOURS}
R_SUP = {pid: support_radius(TEMPLATES[pid], 0.2) for pid in NEIGHBOURS}
N_MASS = {pid: int(round(ZOO[pid]["mass_kDa"] * 1000 / 0.81 / VOXEL_A**3)) for pid in TARGETS}
print("mass-based mask sizes (voxels):", {SHORT[p]: n for p, n in N_MASS.items()})


def crowded_box(target: str, rng: np.random.Generator, max_neighbours: int = 3):
    """Noise-free crowded box: centred target + 1..max_neighbours off-centre neighbours.

    Returns (clean density, target mask [bool], number of neighbours).
    """
    tgt = rotate_volume(TEMPLATES[target], random_rotation(rng), rng.uniform(-1, 1, 3))
    thr = np.sort(tgt.ravel())[-N_MASS[target]]               # mass-based contour level
    clean, n_nb = tgt.copy(), int(rng.integers(1, max_neighbours + 1))
    for _ in range(n_nb):
        pid = NEIGHBOURS[rng.integers(len(NEIGHBOURS))]
        direction = rng.normal(size=3); direction /= np.linalg.norm(direction)
        offset = direction * (R_SUP[target] + R_SUP[pid] * rng.uniform(0.7, 1.1))   # just touching the target
        clean += rotate_volume(TEMPLATES[pid], random_rotation(rng), offset)
    return clean.astype(np.float32), tgt >= thr, n_nb


def image_box(clean: np.ndarray, rng: np.random.Generator, snr: float = 0.1, extra=None) -> np.ndarray:
    """Missing wedge + white noise (SNR measured on `clean`), then normalise.

    `extra` (e.g. a gold bead) is added to the density before imaging but does not change the noise level.
    """
    sigma = np.sqrt(clean.var() / snr)
    v = clean if extra is None else clean + extra
    v = apply_fourier_mask(v, WEDGE) + rng.normal(0, sigma, v.shape)
    return normalize(v)


def simulate_crowded_set(n_per_class: int, rng: np.random.Generator, snr: float = 0.1):
    """Cached dataset: clean boxes, masks, labels and the imaged (noisy) boxes X [N, 1, 24, 24, 24]."""
    clean, masks, y = [], [], []
    for k, pid in enumerate(TARGETS):
        for _ in range(n_per_class):
            c, m, _ = crowded_box(pid, rng)
            clean.append(c); masks.append(m); y.append(k)
    clean, masks, y = np.stack(clean), np.stack(masks), np.array(y)
    X = np.stack([image_box(c, rng, snr) for c in clean])[:, None]
    return dict(clean=clean, mask=masks, y=y, X=X)


t0 = time.time()
TRAIN = simulate_crowded_set(150, np.random.default_rng(SEED))
TEST = simulate_crowded_set(40, np.random.default_rng(SEED + 1))
print(f"train {TRAIN['X'].shape}, test {TEST['X'].shape} simulated in {time.time() - t0:.1f} s; "
      f"mean target fraction of the box: {TRAIN['mask'].mean() * 100:.1f} %")

In [ ]:
# ── 0.2 Look at a few crowded boxes ──
fig, axes = plt.subplots(3, 4, figsize=(7.2, 5.6))
for j, k in enumerate(range(len(TARGETS))):
    i = int(np.where(TRAIN["y"] == k)[0][0])
    axes[0, j].imshow(TRAIN["clean"][i].sum(0)); axes[0, j].set_title(f"{NAMES[k]}\nclean projection", fontsize=8)
    axes[1, j].imshow(TRAIN["X"][i, 0, BOX // 2]); axes[1, j].set_title("SNR 0.1, xy slice", fontsize=8)
    axes[1, j].contour(TRAIN["mask"][i, BOX // 2], levels=[0.5], colors="r", linewidths=0.8)
    axes[2, j].imshow(TRAIN["mask"][i].sum(0), cmap="Reds"); axes[2, j].set_title("target mask (projection)", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Crowded subtomograms: a centred target, neighbours poking in (red contour = target mask)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each box contains the target **and** pieces of other molecules — exactly what a picker hands
you in a crowded cytoplasm. The red contour (mass-based mask) is the answer a segmentation network should learn; in the
noisy slice it is very hard to see by eye.

---
## 1 · Looking inside a classifier: Grad-CAM vs Respond-CAM

### 📍 Where we are
Module 16 trained 3D CNNs and used their pre-softmax outputs as features. Here we ask *which voxels* drove a decision,
then turn the same machinery into segmentation. We use one network family throughout: a small **3D encoder** (stride-2
stem → 12³ → pool → 6³ feature maps $A$) followed by different **heads**.

### 1.1 Class activation maps in three steps

Let $A^{(l)}$ be the $l$-th feature map (channel) of the last convolutional layer — a 6 × 6 × 6 grid of activations —
and $y^{(c)}$ the **pre-softmax score** of class $c$. A *class activation map* is a weighted sum of the feature maps,

$$L^{(c)} \;=\; \sum_l \alpha_l^{(c)}\, A^{(l)},$$

where the weight $\alpha_l^{(c)}$ says how much channel $l$ matters for class $c$. Methods differ only in $\alpha$:

| method | weight $\alpha_l^{(c)}$ | in words |
|---|---|---|
| **CAM** (Zhou *et al.* 2016) | the output-layer weight of channel $l$ | needs a special architecture (global average pooling → one linear layer) |
| **Grad-CAM** (Selvaraju *et al.* 2017) | $\dfrac{1}{Z}\sum_{i,j,k} \dfrac{\partial y^{(c)}}{\partial A^{(l)}_{ijk}}$ | the *average gradient* of the score over the map |
| **Respond-CAM** (Zhao *et al.* 2018) | $\dfrac{\sum_{i,j,k} A^{(l)}_{ijk}\,\partial y^{(c)}/\partial A^{(l)}_{ijk}}{\sum_{i,j,k} A^{(l)}_{ijk} + \epsilon}$ | the gradient averaged **with the activations as weights** |

*In words:* Grad-CAM asks "if I nudged this channel everywhere, how would the score change on average?"; Respond-CAM
asks the same question but only where the channel is actually **active** — where the network's evidence really is.
Respond-CAM applies no ReLU to the map, so it can be negative ("evidence against class $c$").

**The sum-to-score property** (Respond-CAM paper, Eq. 3). If the feature maps feed **one** fully connected layer,
$y^{(c)} = b^{(c)} + \sum_{l,i,j,k} W^{(c)}_{lijk} A^{(l)}_{ijk}$, then $\partial y^{(c)}/\partial A^{(l)}_{ijk} = W^{(c)}_{lijk}$
and the activation-weighted average telescopes:

$$\sum_{i,j,k} L^{(c)}_{ijk} \;=\; \sum_l \alpha^{(c)}_l \sum_{i,j,k} A^{(l)}_{ijk} \;=\; \sum_{l,i,j,k} A^{(l)}_{ijk} W^{(c)}_{lijk} \;=\; y^{(c)} - b^{(c)} .$$

*In words:* the voxels of a Respond-CAM add up **exactly** to the class score (minus the bias). The map is then a true
*decomposition* of the decision into voxel contributions. With deeper heads the property is only approximate.

### 1.2 A shared encoder with interchangeable heads

The network below is used in every section: `heads=("cls",)` gives a classifier (one linear layer on the flattened
6³ feature maps — exactly the setting of the sum-to-score property), `("seg",)` an encoder–decoder (Section 3), and
`("cls", "seg", "rec")` the multi-task network (Section 4).

In [ ]:
# ── 1.2 The network family and one training function for all heads ──
def conv_bn_relu(c_in, c_out, stride=1):
    """3×3×3 convolution → batch normalisation → ReLU."""
    return nn.Sequential(nn.Conv3d(c_in, c_out, 3, stride=stride, padding=1), nn.BatchNorm3d(c_out), nn.ReLU())


class SubtomoNet(nn.Module):
    """3D encoder (24³ → 12³ → 6³) with optional heads: 'cls' (single linear layer), 'seg' and 'rec' (decoder)."""

    def __init__(self, n_classes=4, heads=("cls",), w=8, skip=True):
        super().__init__()
        self.heads, self.skip = heads, skip
        self.stem = conv_bn_relu(1, w, stride=2)                         # 24³ → 12³
        self.enc2 = conv_bn_relu(w, 2 * w)                               # 12³  (skip connection source)
        self.enc3 = nn.Sequential(nn.MaxPool3d(2), conv_bn_relu(2 * w, 4 * w))   # 6³: the maps A
        if "cls" in heads:
            self.cls_head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(4 * w * 216, n_classes))
        if "seg" in heads or "rec" in heads:
            self.up = nn.ConvTranspose3d(4 * w, 2 * w, 2, stride=2)      # 6³ → 12³
            self.dec = conv_bn_relu(2 * w, w)
            self.seg_out = nn.Conv3d(w, 2, 1)                            # background / target logits
            self.rec_out = nn.Conv3d(w, 1, 1)                            # coarse density (recovery)

    def encode(self, x):
        """Return (A, f2): the 6³ feature maps and the 12³ skip features."""
        f2 = self.enc2(self.stem(x))
        return self.enc3(f2), f2

    def forward(self, x):
        A, f2 = self.encode(x)
        out = {}
        if "cls" in self.heads:
            out["cls"] = self.cls_head(A)
        if "seg" in self.heads or "rec" in self.heads:
            d = self.dec(self.up(A) + (f2 if self.skip else 0))
            up = lambda t: F.interpolate(t, scale_factor=2, mode="trilinear", align_corners=False)  # 12³ → 24³
            out["seg"], out["rec"] = up(self.seg_out(d)), up(self.rec_out(d))
        return out


def wedge_safe_flip(tensors, gen):
    """Apply the same random 180° rotation (about z, y or x, or none) to aligned tensors [B, C, z, y, x]."""
    choice = torch.randint(0, 4, (len(tensors[0]),), generator=gen)
    outs = [t.clone() for t in tensors]
    for c, dims in {1: (3, 4), 2: (2, 4), 3: (2, 3)}.items():
        sel = choice == c
        if sel.any():
            for o, t in zip(outs, tensors):
                o[sel] = torch.flip(t[sel], dims)
    return outs


def soft_dice_loss(seg_logits, mask):
    """1 − soft Dice of the target channel (mask [B, 1, z, y, x] in {0, 1})."""
    p = F.softmax(seg_logits, 1)[:, 1:2]
    inter = (p * mask).sum((1, 2, 3, 4))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3, 4)) + mask.sum((1, 2, 3, 4)) + 1)).mean()


def train_net(model, X, y=None, masks=None, rec=None, epochs=6, batch=32, lr=3e-3,
              w_cls=1.0, w_seg=10.0, w_rec=1.0, seed=SEED, verbose=True):
    """Train any combination of heads; only the targets that are given contribute to the loss.

    L = w_cls·CE(class) + w_seg·[CE(voxels) + (1 − soft Dice)] + w_rec·MSE(recovery target).
    Wedge-safe 180° rotations are applied identically to volumes, masks and recovery targets.
    """
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(X) / batch))
    Xt = torch.as_tensor(X)
    Mt = None if masks is None else torch.as_tensor(masks[:, None], dtype=torch.float32)
    Rt = None if rec is None else torch.as_tensor(rec[:, None], dtype=torch.float32)
    Yt = None if y is None else torch.as_tensor(y)
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(X), generator=gen); total = 0.0
        for i in range(0, len(X), batch):
            idx = perm[i:i + batch]
            parts = [Xt[idx]] + [t[idx] for t in (Mt, Rt) if t is not None]
            parts = wedge_safe_flip(parts, gen)
            xb = parts.pop(0).to(DEVICE)
            mb = parts.pop(0).to(DEVICE) if Mt is not None else None
            rb = parts.pop(0).to(DEVICE) if Rt is not None else None
            out, loss = model(xb), 0.0
            if Yt is not None:
                loss = loss + w_cls * F.cross_entropy(out["cls"], Yt[idx].to(DEVICE))
            if mb is not None:
                loss = loss + w_seg * (F.cross_entropy(out["seg"], mb[:, 0].long()) + soft_dice_loss(out["seg"], mb))
            if rb is not None:
                loss = loss + w_rec * F.mse_loss(out["rec"], rb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(idx)
        if verbose:
            print(f"  epoch {ep + 1:2d}/{epochs}  loss {total / len(X):.3f}  ({time.time() - t0:.0f} s)")
    return time.time() - t0


@torch.no_grad()
def predict(model, X, batch=64):
    """Run the model in eval mode; returns dict with 'cls' logits [N, C] and/or 'seg' masks / 'rec' maps [N, z, y, x]."""
    model.eval()
    res = {}
    for i in range(0, len(X), batch):
        out = model(torch.as_tensor(X[i:i + batch]).to(DEVICE))
        for k, v in out.items():
            v = v.argmax(1) if k == "seg" else (v[:, 0] if k == "rec" else v)
            res.setdefault(k, []).append(v.cpu().numpy())
    return {k: np.concatenate(v) for k, v in res.items()}


for heads in [("cls",), ("seg",), ("cls", "seg", "rec")]:
    m = SubtomoNet(heads=heads)
    out = m(torch.zeros(2, 1, BOX, BOX, BOX))
    print(f"heads={str(heads):22s} {sum(p.numel() for p in m.parameters()):7,d} weights → "
          + ", ".join(f"{k}: {list(v.shape)}" for k, v in out.items()))

### 1.3 Train a classifier on image-level labels only

The classifier sees only "this box contains a capped proteasome" — never a mask. (We will reuse this exact network in
Section 5 to *generate* masks.)

In [ ]:
# ── 1.3 Train the classifier (image labels only) ──
torch.manual_seed(SEED)
clf = SubtomoNet(heads=("cls",))
sec = train_net(clf, TRAIN["X"], y=TRAIN["y"], epochs=6)
pred_test = predict(clf, TEST["X"])["cls"].argmax(1)
acc_clf = (pred_test == TEST["y"]).mean()
print(f"test accuracy {acc_clf:.2f} (chance 0.25), trained in {sec:.0f} s")
assert acc_clf > 0.5, "the classifier should be well above chance"

### 1.4 Implement Grad-CAM and Respond-CAM with autograd

PyTorch computes $\partial y^{(c)}/\partial A$ for us: we run the encoder, mark $A$ as "requires gradient", evaluate
the head, and call `torch.autograd.grad`. One function covers both methods and **any** head (we will plug in a deeper
head in 1.6).

In [ ]:
# ── 1.4 Grad-CAM and Respond-CAM for every class ──
def class_activation_maps(model, X, head=None, eps=1e-10, batch=100):
    """Return (gradcam, respondcam, scores): maps [N, C, 6, 6, 6] for every class c, and scores y [N, C].

    Grad-CAM weight   α_l = mean over voxels of ∂y_c/∂A_l
    Respond-CAM weight α_l = Σ A_l ∂y_c/∂A_l / (Σ A_l + ε)          (no ReLU on either map)
    """
    model.eval()
    head = head or model.cls_head
    out_g, out_r, out_s = [], [], []
    for i in range(0, len(X), batch):                          # batches keep memory small
        with torch.no_grad():
            A, _ = model.encode(torch.as_tensor(X[i:i + batch]).to(DEVICE))
        A = A.requires_grad_(True)
        scores = head(A)
        grad_maps, resp_maps = [], []
        for c in range(scores.shape[1]):
            g = torch.autograd.grad(scores[:, c].sum(), A, retain_graph=True)[0]      # ∂y_c/∂A for every box
            alpha_grad = g.mean(dim=(2, 3, 4))
            alpha_resp = (A * g).sum(dim=(2, 3, 4)) / (A.sum(dim=(2, 3, 4)) + eps)
            grad_maps.append((alpha_grad[:, :, None, None, None] * A).sum(1))
            resp_maps.append((alpha_resp[:, :, None, None, None] * A).sum(1))
        out_g.append(torch.stack(grad_maps, 1).detach().cpu()); out_r.append(torch.stack(resp_maps, 1).detach().cpu())
        out_s.append(scores.detach().cpu())
    return torch.cat(out_g).numpy(), torch.cat(out_r).numpy(), torch.cat(out_s).numpy()


G_test, R_test, y_test = class_activation_maps(clf, TEST["X"])
print("map shape:", R_test.shape, "(test boxes, classes, 6, 6, 6)")

### 1.5 Verify the sum-to-score property

For every test box and every class we compare the sum of the map with the score minus the bias, using the paper's two
metrics: the mean absolute (L1) error, and **Kendall's τ** between the ranking of the class scores and the ranking of
the map sums (τ = 1: same order; τ = −1: reversed).

In [ ]:
# ── 1.5 Do the maps add up to the scores? ──
def sum_to_score_report(gradcam, respcam, scores, bias, label):
    """Print L1 error |Σ L_c − (y_c − b_c)| and mean Kendall τ between score ranking and map-sum ranking."""
    target = scores - bias
    for name, maps in [("Grad-CAM", gradcam), ("Respond-CAM", respcam)]:
        sums = maps.sum(axis=(2, 3, 4))
        l1 = np.abs(sums - target).mean()
        tau = np.nanmean([kendalltau(s, t)[0] for s, t in zip(sums, scores)])
        print(f"{label:28s} {name:12s}: mean |Σ L − (y − b)| = {l1:9.2e}   Kendall τ = {tau:+.2f}")
    return target


bias = clf.cls_head[-1].bias.detach().cpu().numpy()
target = sum_to_score_report(G_test, R_test, y_test, bias, "single linear head")
err_resp = np.abs(R_test.sum(axis=(2, 3, 4)) - target).max()
print(f"largest Respond-CAM error over {target.size} (box, class) pairs: {err_resp:.1e}  (scores are of order "
      f"{np.abs(y_test).mean():.0f})")
assert err_resp < 1e-3 * max(1.0, np.abs(y_test).max()), "Respond-CAM should sum to the score minus the bias"

🔍 **What just happened?** For a single linear head, the Respond-CAM voxels add up to the class score minus its bias
to floating-point precision, for every box and every class; Grad-CAM's sums are off by much more and do not even rank
the classes reliably (τ well below 1). The derivation in 1.1 is not an approximation — it is algebra.

### 1.6 When the property is only approximate: a deeper head

Most real classifiers have two or more fully connected layers. Let us train a two-layer head
(Linear → ReLU → Linear) on the **frozen** feature maps of the same encoder (cheap: we only train the head) and repeat
the test. Then we remove the *hidden* bias and test again.

In [ ]:
# ── 1.6 Two-layer heads: with and without a hidden bias ──
with torch.no_grad():
    A_train = torch.cat([clf.encode(torch.as_tensor(TRAIN["X"][i:i + 100]).to(DEVICE))[0]
                         for i in range(0, len(TRAIN["X"]), 100)])


def train_head(hidden_bias: bool, steps=300, seed=SEED):
    """Train Linear(·, 64) → ReLU → Linear(64, 4) on frozen 6³ feature maps."""
    torch.manual_seed(seed)
    head = nn.Sequential(nn.Flatten(), nn.Linear(A_train[0].numel(), 64, bias=hidden_bias), nn.ReLU(),
                         nn.Linear(64, len(TARGETS))).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
    yt = torch.as_tensor(TRAIN["y"]).to(DEVICE)
    for s in range(steps):
        idx = torch.randint(0, len(A_train), (64,))
        loss = F.cross_entropy(head(A_train[idx]), yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return head.eval()


for hb in [True, False]:
    head2 = train_head(hidden_bias=hb)
    G2, R2, y2 = class_activation_maps(clf, TEST["X"], head=head2)
    acc2 = (y2.argmax(1) == TEST["y"]).mean()
    sum_to_score_report(G2, R2, y2, head2[-1].bias.detach().cpu().numpy(),
                        f"2-layer head, hidden bias={hb}")
    print(f"{'':28s} (test accuracy with this head: {acc2:.2f})")

🔍 **What just happened?** With a hidden layer, Respond-CAM's sums no longer match the scores exactly — the error is
small here but no longer at floating-point level — consistent with the paper, which states that the property is
approximate for several FC layers and gives no formal error bound. It still ranks the classes far better than Grad-CAM.
The surprise is the last line: **without the hidden bias the property is exact again.**

💡 **Why?** (Our derivation, not from the paper.) A network built only from linear layers *without biases* and ReLUs
is *positively homogeneous*: scaling the input by $a > 0$ scales the output by $a$. For such functions **Euler's
theorem** gives $\sum_x x\,\partial f/\partial x = f$, and Respond-CAM's sum is exactly $\sum A\,\partial y/\partial A$.
Every hidden bias breaks homogeneity and adds an error term equal to that bias's contribution to the score.

### 1.7 What do the maps look like?

In [ ]:
# ── 1.7 Activation maps of the true class, upsampled to the box ──
def upsample_maps(maps6):
    """Trilinear upsampling of 6³ maps [N, 6, 6, 6] to the 24³ box."""
    t = torch.as_tensor(maps6)[:, None].float()
    return F.interpolate(t, size=(BOX,) * 3, mode="trilinear", align_corners=False)[:, 0].numpy()


fig, axes = plt.subplots(3, 4, figsize=(7.6, 5.8))
for j, k in enumerate(range(len(TARGETS))):
    i = int(np.where(TEST["y"] == k)[0][0])
    g = upsample_maps(G_test[i:i + 1, k])[0]
    r = upsample_maps(R_test[i:i + 1, k])[0]
    zc = int(np.argmax(TEST["mask"][i].sum((1, 2))))                 # slice with most target voxels
    axes[0, j].imshow(TEST["X"][i, 0, zc]); axes[0, j].set_title(f"{NAMES[k]} (z = {zc})", fontsize=8)
    for row, (mp, nm) in enumerate([(g, "Grad-CAM"), (r, "Respond-CAM")], start=1):
        lim = np.abs(mp).max() + 1e-9
        axes[row, j].imshow(mp[zc], cmap="RdBu_r", vmin=-lim, vmax=lim)
        axes[row, j].set_title(nm, fontsize=8)
    for ax in axes[:, j]:
        ax.contour(TEST["mask"][i, zc], levels=[0.5], colors="k", linewidths=0.8); ax.axis("off")
fig.suptitle("Maps for the true class (red = evidence for, blue = against; black = true target outline)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Respond-CAM puts positive evidence (red) on and around the target. Grad-CAM, for the same
network and the same boxes, is often **negative** (blue) exactly where the molecule is. The reason is instructive: our
classifier learned "centre–surround" weights — density in the middle of the box counts *for* the class, density near
the edges (neighbours!) counts *against* it. Grad-CAM averages each channel's gradient **uniformly** over the box, so
the larger surround wins and a channel that fires on the molecule receives a negative weight. Respond-CAM averages the
gradient **where the channel is active**, i.e. on the molecule. This is the failure the Respond-CAM paper documented
(and the reason its sums track the scores while Grad-CAM's do not: τ ≈ 0 above). Both maps are coarse (6³ cells of
6 nm, smoothly upsampled): a CAM tells you *roughly where* the evidence is, not the molecule's outline.

### 📄 Paper connection — Respond-CAM (Zhao, Zhou *et al.* 2018)

**Paper**: Zhao G, Zhou B, Wang K, Jiang R, Xu M (Zhao and Zhou co-first). *Respond-CAM: Analyzing deep models for 3D
imaging data by visualizations.* MICCAI 2018, LNCS, pp. 485–492. [arXiv:1806.00102](https://arxiv.org/abs/1806.00102)

- **Question it answers:** Grad-CAM heat maps of 3D subtomogram CNNs are noisy and unrelated to the class score — can
  we build a map whose voxels add up to the score?
- **What the paper actually does:** weights each feature map by an *activation-weighted* average of its gradients
  (Eq. 1), sums the weighted maps (Eq. 2, no ReLU), proves the sum-to-score property
  $y^{(c)} = b^{(c)} + \sum L^{(c)}$ for a CAM layer feeding a single FC layer (Eq. 3; approximate otherwise, without a
  formal error bound), and evaluates on two 3D CNNs (one from Xu *et al.* 2017, one modified to > 96 % accuracy) trained
  on simulated 40³ subtomograms of 23 classes (±60°, SNR ∞ and 0.1), with a qualitative check on natural images.
- **What our toy keeps:** both formulas, autograd implementation, the L1 error and Kendall's τ metrics, single- vs
  multi-layer heads.
- **What our toy leaves out:** the paper's CNNs and 23-class data; the natural-image experiments.
- **Result reported:** sum-to-score L1 error 0.54–1.01 for Respond-CAM vs 8.1–21.1 for Grad-CAM; Kendall's τ ≥ 0.975 for
  Respond-CAM vs −0.12 to 0.82 for Grad-CAM, across both CNNs and both SNRs.
- **Code:** AITom `aitom/segmentation/respond_cam/` (`respond_cam.py`: `weights = sum(A*grad)/sum(A+1e-10)`,
  `cam = sum(A*weights)`).

---
## 2 · Right for the wrong reason: a gold-bead shortcut

🧫 Colloidal **gold beads** (5–10 nm) are added to many cryo-ET samples as fiducial markers for tilt-series alignment
(Module 07). They are extremely dense and bright. Imagine that all your capped-proteasome training examples came from
one tomogram that happened to be full of beads, so most boxes of that class contain a bead fragment. A network will
happily learn "bead ⇒ capped proteasome" — the easiest rule that fits the training labels. Such **shortcut learning**
(the "Clever Hans" effect, after a horse that seemed to count but actually read its trainer's body language) is
invisible in accuracy on similarly biased test data. Let us create it on purpose and see whether activation maps
reveal it.

In [ ]:
# ── 2.1 A biased training set: every capped-proteasome box contains a gold bead ──
BEAD_CLASS = TARGETS.index("6utj")
GRID = np.stack(np.meshgrid(*[np.arange(BOX)] * 3, indexing="ij"))          # voxel coordinates [3, z, y, x]


def gold_bead(rng, sigma=1.2, amplitude=10.0):
    """A small sphere (≈ 5–6 nm) of gold — far denser than protein — near the box edge. Returns (density, centre)."""
    direction = rng.normal(size=3); direction /= np.linalg.norm(direction)
    c = (BOX - 1) / 2 + direction * rng.uniform(8.5, 10.5)
    d2 = ((GRID - c[:, None, None, None]) ** 2).sum(0)
    return (amplitude * np.exp(-d2 / (2 * sigma**2))).astype(np.float32), c


bead_rng = np.random.default_rng(SEED + 5)
X_biased = np.stack([image_box(c, bead_rng, 0.1, extra=gold_bead(bead_rng)[0] if y == BEAD_CLASS else None)
                     for c, y in zip(TRAIN["clean"], TRAIN["y"])])[:, None]
test_beads = [gold_bead(bead_rng) for _ in TEST["y"]]                         # a bead in EVERY test box
X_test_beads = np.stack([image_box(c, bead_rng, 0.1, extra=b[0]) for c, b in zip(TEST["clean"], test_beads)])[:, None]
bead_centres = np.stack([b[1] for b in test_beads])
torch.manual_seed(SEED)
clf_biased = SubtomoNet(heads=("cls",))
sec = train_net(clf_biased, X_biased, y=TRAIN["y"], epochs=6, verbose=False)
del X_biased                                                                  # free memory
print(f"trained the biased classifier in {sec:.0f} s")
for label, X in [("clean test boxes (no beads)", TEST["X"]), ("test boxes, a bead in EVERY box", X_test_beads)]:
    for name, m in [("normal classifier", clf), ("bead-trained classifier", clf_biased)]:
        p = predict(m, X)["cls"].argmax(1)
        rec = (p[TEST["y"] == BEAD_CLASS] == BEAD_CLASS).mean()
        print(f"{label:33s} | {name:24s}: accuracy {(p == TEST['y']).mean():.2f}; called '{NAMES[BEAD_CLASS]}': "
              f"{(p == BEAD_CLASS).mean() * 100:3.0f} % of boxes (true share 25 %); recall of {NAMES[BEAD_CLASS]} {rec:.2f}")

In [ ]:
# ── 2.2 The activation maps give the shortcut away ──
def bead_evidence(model):
    """Positive Respond-CAM evidence for the bead class in bead-containing test boxes: total, and share within 6 nm of the bead."""
    _, R_, _ = class_activation_maps(model, X_test_beads)
    maps = np.clip(upsample_maps(R_[:, BEAD_CLASS]), 0, None)
    near = ((GRID[None] - bead_centres[:, :, None, None, None]) ** 2).sum(1) < 16
    other = TEST["y"] != BEAD_CLASS                                     # boxes that contain NO capped proteasome
    total = maps.sum((1, 2, 3))
    share = (maps * near).sum((1, 2, 3)) / (total + 1e-9)
    return float(total[other].mean()), float(share[other].mean()), maps


tot_norm, share_norm, maps_norm = bead_evidence(clf)
tot_bead, share_bead, maps_bead = bead_evidence(clf_biased)
print(f"boxes WITHOUT a capped proteasome but with a bead — total positive evidence for '{NAMES[BEAD_CLASS]}':")
print(f"  normal network {tot_norm:7.1f} | bead-trained network {tot_bead:7.1f}  (×{tot_bead / tot_norm:.0f})")
print(f"  share of it within 6 nm of the bead: normal {100 * share_norm:.0f} %, bead-trained {100 * share_bead:.0f} % "
      f"(that sphere is {100 * 268 / BOX**3:.0f} % of the box)")

i = int(np.where(TEST["y"] == TARGETS.index("3j9i"))[0][0])               # a 20S proteasome WITH a bead
zb = int(round(bead_centres[i][0]))
fig, axes = plt.subplots(1, 3, figsize=(7.8, 2.9))
axes[0].imshow(X_test_beads[i, 0, zb]); axes[0].set_title(f"20S proteasome + bead\n(slice z = {zb})", fontsize=8)
for ax, (mp, nm) in zip(axes[1:], [(maps_norm[i], "normal"), (maps_bead[i], "bead-trained")]):
    ax.imshow(mp[zb], cmap="Reds", vmin=0, vmax=mp.max() + 1e-9)
    ax.set_title(f"{nm} network: positive Respond-CAM\nfor '{NAMES[BEAD_CLASS]}'", fontsize=8)
for ax in axes:
    ax.plot(bead_centres[i][2], bead_centres[i][1], "c+", ms=12, mew=2); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without beads, the bead-trained classifier rarely says "capped proteasome" — its recall for
that class collapses, because it learned to look for the bead instead of the cap. With a bead in every box it says
"capped proteasome" more often than the normal network, for boxes of any class. The activation maps raise the alarm
without a specially designed test set: in boxes that contain *no* capped proteasome, the bead-trained network still
finds an order of magnitude more "capped proteasome" evidence than the normal network — evidence for a molecule that
is not there. Where exactly? The maps are too coarse to say: each CAM cell is 6 nm wide, the bead only ~2 nm, so the
evidence is smeared over the side of the box where the bead (cyan cross) sits, and the share inside a 6 nm sphere
around the bead is only modestly above that sphere's share of the volume. A CAM tells you *that* something is wrong and
roughly *where*; finding the culprit still takes a look at the data. In real projects the "bead" may be
subtler: the defocus or ice thickness of one tomogram, the box-edge artefacts of one picker. **Always look at the maps
of a few correct predictions, not only of the errors.**

✏️ **Your turn.** Put beads in only 50 % of the capped-proteasome training boxes (edit 2.1: `if y == BEAD_CLASS and
bead_rng.random() < 0.5`). Is the shortcut still learned?

---
## 3 · Voxel-wise semantic segmentation (Liu C *et al.* 2018, ICIP)

**Semantic segmentation** assigns a class to every voxel — here *target* vs *everything else* (background and
neighbours). An **encoder–decoder** first compresses the box into coarse feature maps (the encoder we already have) and
then **up-samples** them back to full resolution (the decoder: a transposed convolution 6³ → 12³, a convolution, a 1×1×1
convolution to two class scores, and a final trilinear interpolation to 24³). A **skip connection** adds the
fine-grained 12³ encoder features to the decoder, restoring detail lost by pooling (the idea behind U-Net).

**Loss.** Per-voxel cross-entropy. Because the target fills only 1–6 % of the box, we add a **soft Dice** term,
$1 - \frac{2\sum p\,g + 1}{\sum p + \sum g + 1}$ (with predicted target probabilities $p$ and ground truth $g$), a
common remedy for voxel-level class imbalance (the SSN3D paper itself used cross-entropy only).

**Metrics** for a predicted mask $P$ and true mask $G$:

$$\text{Dice} = \frac{2\,|P \cap G|}{|P| + |G|}, \qquad \text{IoU} = \frac{|P \cap G|}{|P \cup G|}, \qquad
\text{Dice} = \frac{2\,\text{IoU}}{1 + \text{IoU}} .$$

*In words:* both measure overlap (1 = perfect, 0 = none); Dice is always the larger of the two. "mIoU" in papers is
the IoU averaged over classes (here we report the target-class IoU).

In [ ]:
# ── 3.1 Train the encoder–decoder on all 600 training masks ──
def overlap_scores(pred, true):
    """Mean Dice and IoU over boxes for boolean masks [N, z, y, x]."""
    pred, true = pred.astype(bool), true.astype(bool)
    inter = (pred & true).sum((1, 2, 3)); union = (pred | true).sum((1, 2, 3))
    dice = 2 * inter / np.maximum(pred.sum((1, 2, 3)) + true.sum((1, 2, 3)), 1)
    return float(dice.mean()), float((inter / np.maximum(union, 1)).mean())


torch.manual_seed(SEED)
seg_full = SubtomoNet(heads=("seg",))
sec = train_net(seg_full, TRAIN["X"], masks=TRAIN["mask"], epochs=8)
seg_pred = predict(seg_full, TEST["X"])["seg"]
dice_full, iou_full = overlap_scores(seg_pred, TEST["mask"])
print(f"fully supervised segmentation: test Dice {dice_full:.2f}, IoU {iou_full:.2f} ({sec:.0f} s)")
for k in range(len(TARGETS)):
    d, u = overlap_scores(seg_pred[TEST["y"] == k], TEST["mask"][TEST["y"] == k])
    print(f"   {NAMES[k]:12s}: Dice {d:.2f}  IoU {u:.2f}")
assert dice_full > 0.3, "the segmentation network should find the target"

In [ ]:
# ── 3.2 Look at predictions next to the truth ──
fig, axes = plt.subplots(2, 4, figsize=(7.6, 4))
for j, k in enumerate(range(len(TARGETS))):
    i = int(np.where(TEST["y"] == k)[0][1])
    zc = int(np.argmax(TEST["mask"][i].sum((1, 2))))
    axes[0, j].imshow(TEST["X"][i, 0, zc]); axes[0, j].set_title(f"{NAMES[k]}: input slice", fontsize=8)
    axes[1, j].imshow(TEST["clean"][i, zc])
    axes[1, j].contour(TEST["mask"][i, zc], levels=[0.5], colors="lime", linewidths=1)
    axes[1, j].contour(seg_pred[i, zc], levels=[0.5], colors="r", linewidths=1, linestyles="--")
    d, _ = overlap_scores(seg_pred[i:i + 1], TEST["mask"][i:i + 1])
    axes[1, j].set_title(f"clean density; Dice {d:.2f}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Green = true target mask, red dashed = network prediction (bottom row shows the clean density for reference)",
             fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From noisy boxes in which most of us see nothing, a ~40,000-weight network outlines the
central molecule and ignores the neighbours — useful for masking before alignment (Module 13) or averaging. Small
targets (RuBisCO, ≈ 100 voxels) are hardest: a one-voxel error on the boundary is a large fraction of their volume,
which is why Dice/IoU are always lower for small objects.

✏️ **Your turn.** Train `SubtomoNet(heads=("seg",), skip=False)` (no skip connection) with the same call and compare the
Dice. Where do the predicted outlines lose detail?

### 📄 Paper connection — supervised semantic segmentation of subtomograms (Liu C *et al.* 2018, ICIP)

**Paper**: Liu C, Zeng X, Lin R, Liang X, Freyberg Z, Xing E, Xu M. *Deep learning based supervised semantic
segmentation of electron cryo-subtomograms.* IEEE ICIP 2018, pp. 1578–1582. [arXiv:1802.04087](https://arxiv.org/abs/1802.04087)

- **Question it answers:** in crowded cells a subtomogram contains fragments of neighbours that bias alignment and
  averaging — can a network segment only the target, voxel by voxel?
- **What the paper actually does:** compares three 3D fully convolutional networks trained with categorical
  cross-entropy: **SSN3D-v1** (VGG-like encoder, two fully convolutional layers with dropout, 3 upsampling blocks;
  2.04 M parameters), **SSN3D-v2** (v1 + FCN-style skip connections) and **SSN3D-ED** (a symmetric encoder–decoder;
  0.90 M parameters). Data: simulated tomograms (600 × 600 × 300, ±60°, ≈ 10,000 macromolecules of 22 complexes) →
  3,380 subtomograms at 40³ (3,042 train / 338 test) at SNR 0.5, 0.06 and 0.01.
- **What our toy keeps:** a crowded target-vs-rest segmentation task, an encoder–decoder with a skip connection,
  pixel-overlap metrics.
- **What our toy leaves out:** the three architectures, the 22-class tomograms, leave-one-class-out tests; we add a
  soft-Dice loss term and use mass-based masks (the paper does not describe its mask generation in detail).
- **Result reported:** at SNR 0.5, SSN3D-ED reached 99.35 % pixel accuracy and 86.4 % mIoU (vs 27.4 % for an earlier
  baseline); 73.8 % mIoU at SNR 0.01; every held-out structure in leave-one-class-out tests reached ≥ 72 % mIoU.
- **Code:** no dedicated AITom folder; related code in `aitom/classify/deep/supervised/cnn/mult_task/`.

---
## 4 · Multi-task learning: one encoder, three heads (Liu C *et al.* 2018, BMVC)

🧫 Labelled real subtomograms number in the hundreds, not thousands. Classification, segmentation and structure
recovery all need the same underlying skill — *recognise the molecule's shape through noise* — so why learn them
separately? A **multi-task** network shares one encoder between several heads, so every task's labels train the shared
features. Following DSM-Net, our total loss is

$$\mathcal L = \mathcal L_{cls} + 10\,\mathcal L_{seg} + \mathcal L_{rec}, \qquad
\mathcal L_{rec} = \frac{1}{N}\sum_x \lVert G(x) - H(x)\rVert^2 ,$$

where $H$ is our network's recovery output and $G(x)$ is a **pseudo ground truth produced without any human labels**.
In the paper, $G$ was the response of an unsupervised model. Our stand-in for $G$ is a heavily smoothed copy of the
input box (Gaussian, σ = 2 voxels) — a label-free target that rewards denoising-like features. *In words:* the
recovery head never sees the true density; it is an auxiliary self-supervised task.

We deliberately use a **small** training set — 40 boxes per class — where single-task learning struggles.

In [ ]:
# ── 4.1 Single-task vs multi-task on a small labelled set ──
small = np.concatenate([np.where(TRAIN["y"] == k)[0][:40] for k in range(len(TARGETS))])
Xs, ys, ms = TRAIN["X"][small], TRAIN["y"][small], TRAIN["mask"][small]
rec_target = np.stack([ndimage.gaussian_filter(x[0], 2.0) for x in Xs])      # label-free pseudo ground truth

runs = {}
t0 = time.time()
for name, heads, kw in [("classification only", ("cls",), dict(y=ys)),
                        ("segmentation only", ("seg",), dict(masks=ms)),
                        ("multi-task 1:10:1", ("cls", "seg", "rec"), dict(y=ys, masks=ms, rec=rec_target))]:
    torch.manual_seed(SEED)
    net = SubtomoNet(heads=heads)
    train_net(net, Xs, epochs=12, batch=16, verbose=False, **kw)
    out = predict(net, TEST["X"])
    acc = (out["cls"].argmax(1) == TEST["y"]).mean() if "cls" in out and "cls" in heads else np.nan
    dice = overlap_scores(out["seg"], TEST["mask"])[0] if "seg" in heads else np.nan
    runs[name] = (acc, dice)
    print(f"{name:20s}: test accuracy {acc:.2f} | test Dice {dice:.2f}")
print(f"({time.time() - t0:.0f} s for three trainings on {len(small)} boxes)")

fig, axes = plt.subplots(1, 2, figsize=(7.6, 2.8))
for ax, j, lab in [(axes[0], 0, "classification accuracy"), (axes[1], 1, "segmentation Dice")]:
    vals = [(n, v[j]) for n, v in runs.items() if not np.isnan(v[j])]
    ax.bar([n.replace(" ", "\n", 1) for n, _ in vals], [v for _, v in vals],
           color=["tab:gray", "tab:purple"] if j == 0 else ["tab:gray", "tab:purple"])
    ax.set_ylim(0, 1); ax.set_title(lab, fontsize=9)
    for k, (_, v) in enumerate(vals):
        ax.text(k, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)
fig.suptitle(f"Small training set ({len(small)} boxes): single-task (grey) vs multi-task (purple)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Compare the purple bars with the grey ones. In the paper, sharing the encoder raised
classification accuracy strongly on a small dataset (and made training converge where the single-task classifier
failed), and slightly improved segmentation. In our toy, the multi-task network typically classifies at least as well
as the classification-only network — the dense mask labels tell the shared encoder *where* the evidence is — while
its segmentation can come out *worse* than the segmentation-only network: our encoder is tiny, so the tasks compete for
its capacity, and the 1:10:1 weights were tuned for the paper's network, not ours. Multi-task learning is a trade-off
to be measured, not a free lunch. With 160 training boxes and one seed, differences of a few points are within noise —
rerun with another seed before drawing conclusions.

✏️ **Your turn.** Change the weights to `w_seg=1.0` (1:1:1) in the multi-task call. The paper reports results to be
fairly insensitive to the ratio — is that true here?

### 📄 Paper connection — multi-task DSM-Net (Liu C *et al.* 2018, BMVC)

**Paper**: Liu C, Zeng X, Wang K, Guo Q, Xu M. *Multi-task learning for macromolecule classification, segmentation and
coarse structural recovery in cryo-tomography.* BMVC 2018. [arXiv:1805.06332](https://arxiv.org/abs/1805.06332)

- **Question it answers:** classification, target segmentation and structure recovery each lack labelled data — can
  one network share what they learn?
- **What the paper actually does:** **DSM-Net**, a shared 3D residual backbone (32/64/128-channel residual blocks) with
  three heads — classification (FC-1024, dropout 0.7, softmax over 22 classes), segmentation (decoder with stride-2 3D
  deconvolutions and skip connections, 2-channel softmax) and recovery (same decoder, 1-channel output). Loss
  $L = L_{cls} + 10 L_{seg} + L_{rec}$; the recovery target $G(x)$ is the response of an **unsupervised model**, used
  as pseudo ground truth (not the true density).
- **What our toy keeps:** shared encoder, three heads, the 1:10:1 weighting, a label-free recovery target, comparison
  with single-task training on a small dataset.
- **What our toy leaves out:** the residual backbone, 22 classes, the real rat-neuron data, recovery of an unseen
  structure; our recovery target is a smoothed input, not an unsupervised model's output.
- **Result reported:** on simulated Dataset A, DSM-Net reached 93.75 % classification accuracy vs 76.24 % for the
  classification-only subnet, and segmentation mIoU 89.00 vs 87.07; on experimental Dataset C (rat neurons; 400
  ribosomes + 386 single-capped proteasomes) DSM-Net classified with 100 % accuracy where the single-task classifier
  failed to converge; with GroEL (1KP8) held out of training, the model still grouped and recovered it.
- **Code:** AITom `aitom/classify/deep/supervised/cnn/mult_task/` (its README says it implements only the
  segmentation part).

---
## 5 · Segmentation from image labels only (in the spirit of CIVA-Net, Zhu *et al.* 2021)

🧫 Drawing a 3D mask takes an expert minutes per subtomogram; saying "this box is a ribosome" takes a second. **Weakly
supervised** segmentation learns voxel masks from such **image-level labels**. CIVA-Net's recipe has four ingredients;
our toy implements simplified versions of the first three:

1. **Seeds from a classifier.** Train a classifier on image labels (our `clf` from Section 1) and compute class
   activation maps of the labelled class: coarse "where is the evidence" maps. CIVA-Net used 3D Grad-CAM; Section 1.7
   showed that in our toy Grad-CAM is mostly *negative* on the molecule, so we compare it with Respond-CAM seeds
   (positive part), the lab's own map for 3D data.
2. **Cross-image consensus.** Copies of the same molecule share a shape, so voxels whose *features* agree with the
   features common to a group of same-class boxes are probably on the molecule. We compute a group prototype — the
   CAM-weighted mean feature vector of $N = 8$ same-class boxes — and, for every voxel, the cosine similarity of its
   feature vector to the prototype: a consensus map $P^c$.
3. **Fusion and pseudo-labels.** After rank normalisation (each map replaced by its percentile ranks, so the two are
   on the same 0–1 scale), $M^c = w_1 G^c + w_2 P^c$ with $w_1 = w_2 = \tfrac12$. We then keep the top-$n$ voxels of
   $M^c$, with $n$ given by the molecule's **mass** (known from the image label) — a biological prior instead of a
   hand-tuned threshold. (We fuse the Respond-CAM seeds with the consensus map — and, as a second option, with the
   box's own smoothed density, which costs no labels either.)
4. *(Not implemented.)* CIVA-Net then trains an inter-voxel-affinity network on the pseudo-labels and refines with a 3D
   conditional random field. We simply train our segmentation network on the pseudo-masks.

In [ ]:
# ── 5.1 Seeds (Grad-CAM) and cross-image consensus maps on the training set ──
def rank_normalise(maps):
    """Replace each map's values by their percentile ranks in [0, 1] (per box)."""
    flat = maps.reshape(len(maps), -1)
    ranks = flat.argsort(1).argsort(1).astype(np.float32) / (flat.shape[1] - 1)
    return ranks.reshape(maps.shape)


def top_n_mask(score_maps, labels):
    """Keep, per box, the n highest-scoring voxels, with n = mass-based volume of the labelled class."""
    out = np.zeros(score_maps.shape, bool)
    for i, (s, k) in enumerate(zip(score_maps, labels)):
        thr = np.sort(s.ravel())[-N_MASS[TARGETS[k]]]
        out[i] = s >= thr
    return out


t0 = time.time()
G_tr, R_tr, _ = class_activation_maps(clf, TRAIN["X"])
labelled = (np.arange(len(TRAIN["y"])), TRAIN["y"])
G_seed, R_seed = G_tr[labelled], R_tr[labelled]                          # maps of the LABELLED class [N, 6, 6, 6]
with torch.no_grad():
    feats = torch.cat([clf.encode(torch.as_tensor(TRAIN["X"][i:i + 100]).to(DEVICE))[0]
                       for i in range(0, len(TRAIN["X"]), 100)]).cpu().numpy()       # [N, C, 6, 6, 6]

consensus = np.zeros_like(R_seed)
group_rng = np.random.default_rng(SEED)
N_GROUP = 8
for k in range(len(TARGETS)):
    members = group_rng.permutation(np.where(TRAIN["y"] == k)[0])
    for g0 in range(0, len(members), N_GROUP):
        grp = members[g0:g0 + N_GROUP]
        w = np.clip(R_seed[grp], 0, None)[:, None]                              # CAM weights [n, 1, 6, 6, 6]
        proto = (w * feats[grp]).sum((0, 2, 3, 4)) / (w.sum() + 1e-9)          # group prototype feature [C]
        f = feats[grp]
        cos = np.einsum("nc...,c->n...", f, proto) / (np.linalg.norm(f, axis=1) * np.linalg.norm(proto) + 1e-9)
        consensus[grp] = cos

G_up, R_up, P_up = upsample_maps(G_seed), upsample_maps(R_seed), upsample_maps(consensus)
dens_tr = np.stack([ndimage.gaussian_filter(x[0], 1.0) for x in TRAIN["X"]])     # each box's own smoothed density
centre = np.broadcast_to(-np.sqrt(((GRID - (BOX - 1) / 2) ** 2).sum(0)), dens_tr.shape)   # "the target is in the middle"
pseudo = {"Grad-CAM seeds": top_n_mask(G_up, TRAIN["y"]),
          "Respond-CAM seeds": top_n_mask(R_up, TRAIN["y"]),
          "consensus only": top_n_mask(P_up, TRAIN["y"]),
          "Respond-CAM + consensus": top_n_mask(rank_normalise(R_up) + rank_normalise(P_up), TRAIN["y"]),
          "Respond-CAM + box density": top_n_mask(rank_normalise(R_up) + rank_normalise(dens_tr), TRAIN["y"]),
          "baseline: density + centre, no network": top_n_mask(rank_normalise(dens_tr) + rank_normalise(centre), TRAIN["y"])}
print(f"maps for {len(TRAIN['y'])} training boxes in {time.time() - t0:.0f} s")
for name, pm in pseudo.items():
    d, u = overlap_scores(pm, TRAIN["mask"])
    print(f"pseudo-masks from {name:38s}: IoU with the (hidden) true masks {u:.2f}, Dice {d:.2f}")

In [ ]:
# ── 5.2 Train a segmentation network on pseudo-masks (no true mask is ever used) ──
torch.manual_seed(SEED)
seg_weak = SubtomoNet(heads=("seg",))
sec = train_net(seg_weak, TRAIN["X"], masks=pseudo["Respond-CAM + box density"], epochs=6, verbose=False)
weak_pred = predict(seg_weak, TEST["X"])["seg"]
R_te = R_test[np.arange(len(TEST["y"])), pred_test]                          # test-time Respond-CAM of the PREDICTED class
cam_test_mask = top_n_mask(upsample_maps(R_te), pred_test)
dens_te = np.stack([ndimage.gaussian_filter(x[0], 1.0) for x in TEST["X"]])
baseline_test = top_n_mask(rank_normalise(dens_te) + rank_normalise(np.broadcast_to(centre[0], dens_te.shape)), pred_test)
rows = [("Respond-CAM thresholded (test time)", cam_test_mask),
        ("network trained on pseudo-masks", weak_pred),
        ("fully supervised (Section 3)", seg_pred),
        ("baseline: density + centre, no network", baseline_test)]
print(f"(weakly supervised training: {sec:.0f} s)")
for name, pm in rows:
    d, u = overlap_scores(pm, TEST["mask"])
    print(f"{name:36s}: test IoU {u:.2f}  Dice {d:.2f}")

fig, axes = plt.subplots(1, 5, figsize=(10.5, 2.4))
i = int(np.where(TEST["y"] == 0)[0][2]); zc = int(np.argmax(TEST["mask"][i].sum((1, 2))))
axes[0].imshow(TEST["clean"][i, zc]); axes[0].set_title("clean density", fontsize=8)
for ax, (name, pm) in zip(axes[1:], rows):
    ax.imshow(TEST["clean"][i, zc]); ax.contour(TEST["mask"][i, zc], levels=[0.5], colors="lime", linewidths=1)
    if pm[i, zc].any():
        ax.contour(pm[i, zc], levels=[0.5], colors="r", linewidths=1, linestyles="--")
    ax.set_title(name.split(" (")[0], fontsize=7)
for ax in axes:
    ax.axis("off")
fig.suptitle("Green = true mask, red dashed = prediction (a 70S ribosome test box)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the pseudo-mask lines first. Grad-CAM seeds — CIVA-Net's starting point — are useless
here (they point away from the molecule, Section 1.7; CIVA-Net, too, found raw Grad-CAM poor: mIoU 14.8 % at SNR 0.03).
Respond-CAM seeds and the cross-image consensus are better but still weak in our toy: the maps have 6 nm cells, while a
target fills only a few per cent of the 36 nm box. Adding something free — the box's own smoothed density — turns the
Respond-CAM seeds into usable pseudo-masks, and a network trained on them, without a single voxel label, gets within
reach of full supervision: the same ordering CIVA-Net reports (Grad-CAM ≪ weakly supervised < fully supervised).

⚠️ Now look at the last line of each table: a **baseline with no network at all** — keep the densest voxels near the
box centre, as many as the mass prior allows — beats every CAM-based pseudo-mask, and here even beats our fully
supervised network. It works because our boxes were cut around *centred* targets of *known* mass. That says more about our toy
than about CIVA-Net, but it is the most useful lesson of this section: **always compare a clever method with simple
baselines**, and ask which prior knowledge (here: centring, mass) a "weakly supervised" method smuggles in.

✏️ **Your turn.** Change the fusion weights (e.g. `0.8 * … + 0.2 * …`) or `N_GROUP` and re-run 5.1. Which ingredient
matters most for the pseudo-mask IoU?

### 📄 Paper connection — CIVA-Net (Zhu *et al.* 2021)

**Paper**: Zhu X, Chen J, Zeng X, Liang J, Li C, Liu S, Behpour S, Xu M. *Weakly supervised 3D semantic segmentation
using cross-image consensus and inter-voxel affinity relations.* ICCV 2021, pp. 2834–2844.
[doi:10.1109/ICCV48922.2021.00283](https://doi.org/10.1109/ICCV48922.2021.00283)

- **Question it answers:** can voxel-level segmentation of subtomograms be learned from **image-level class labels
  only**? (Weakly supervised — not unsupervised; cryo-ET data only.)
- **What the paper actually does:** (1) a 3D CNN classifier gives Grad-CAM seeds $G^c$; (2) groups of $N$ same-class
  volumes are encoded and a group consensus embedding is matched against each image's features to give a co-occurrence
  map $P^c$; fused $M^c = w_1 G^c + w_2 P^c$ after rank normalisation; (3) thresholding gives foreground/background
  pseudo-labels; (4) a VoxResNet-based network learns a boundary map and **inter-voxel affinities** from
  different-label and background–background pairs within radius γ = 2 (foreground–foreground pairs excluded because
  structures can have holes), with loss $\mathcal L = \mathcal L_B + \mathcal L_O$; (5) inference picks the best
  affinity proposal and refines it with a 3D dense CRF. Data: simulated 32³ subtomograms of 4 classes (1BXN, 1F1B,
  1YG6 and a SARS-CoV-2 spike) at SNR 0.03 and 0.05, and a real poly-GA neuron dataset (617 ribosomes, 58 26S
  proteasomes, 81 TRiC).
- **What our toy keeps:** CAM seeds (Respond-CAM instead of Grad-CAM, see above), a group-consensus map,
  rank-normalised fusion, pseudo-masks, training on them.
- **What our toy leaves out:** the learned consensus embedding, the affinity network, the CRF; our mass prior for the
  pseudo-mask size is our own addition.
- **Result reported (mIoU %, SNR 0.03 / SNR 0.05):** Grad-CAM 14.8 / 9.7; CIVA-Net + 3D-CRF 39.9 / 38.8; 3D U-Net with
  2D-slice supervision 30.3 / 34.2; fully supervised VoxResNet 77.0 / 78.5. (On the real data, the CIVA-Net value
  appears as 67.8 in one table and 62.7 in another of the extracted text — check the PDF.)
- **Code:** none given in the paper or in AITom.

### 📄 Paper connection — another kind of weak label: label the clusters (Zeng *et al.* 2018)

**Paper**: Zeng X, Leung MR, Zeev-Ben-Mordehai T, Xu M. *A convolutional autoencoder approach for mining features in
cellular electron cryo-tomograms and weakly supervised coarse segmentation.* J Struct Biol 202(2):150–160 (2018).
[doi:10.1016/j.jsb.2017.12.015](https://doi.org/10.1016/j.jsb.2017.12.015) · [arXiv:1706.04970](https://arxiv.org/abs/1706.04970)

- **Idea:** a 3D convolutional autoencoder compresses DoG-picked 32³ subvolumes into 32-number codes; k-means (k = 100)
  groups the codes; each cluster centre is *decoded* into an image and a human tags it as "surface", "large globular"
  or "not selected". Thresholding the decoded centres at 0.5 gives masks that train **EDSS3D**, an encoder–decoder
  (five 3³ conv layers, two pools, two upsamplings, softmax; cross-entropy; Adam, lr 1e-3) for coarse voxel-wise
  segmentation of new tomograms. The only supervision is ~100 cluster-level clicks.
- **Result reported:** on a simulated tomogram, the selected ribosome clusters gave precision 66.0 % and recall 61.3 %;
  surface clusters 73.9 % and 44.7 %. Real data: two COS-7 cell tomograms, assessed visually.
- **Code:** AITom `aitom/classify/deep/unsupervised/autoencoder/` (`autoencoder.py`, `seg_src.py`). The autoencoder
  half of this paper is implemented in Module 19.

---
## 6 · Other ways to find structures: one shot, boxes, point sets and saliency

**One-shot segmentation (COS-Net).** What if a *new* molecule appears and you have **one** annotated example? COS-Net
(Zhou *et al.* 2020) is a **Siamese** network: two weight-sharing 3D encoder–decoder streams process a support example
and a query; the L1 distance between their 512-d feature vectors decides "same class?", while the query stream also
outputs a voxel mask (attention blocks inside, CRF refinement after). Few-shot learning is the topic of Module 19.

**Detection vs segmentation.** A detector outputs **bounding boxes**, not masks. Boxes are cheaper to annotate and
perfectly adequate for organelles such as mitochondria (Li R *et al.* 2019, Faster-RCNN), but a box around an elongated,
tilted molecule contains a lot of non-molecule. The next cell measures how much, on our true masks.

In [ ]:
# ── 6.1 How much of an axis-aligned bounding box is actually molecule? ──
fill = {}
for k in range(len(TARGETS)):
    ratios = []
    for msk in TEST["mask"][TEST["y"] == k]:
        idx = np.argwhere(msk)
        box_vol = np.prod(idx.max(0) - idx.min(0) + 1)
        ratios.append(msk.sum() / box_vol)
    fill[NAMES[k]] = np.mean(ratios)
    print(f"{NAMES[k]:12s}: the molecule fills {100 * fill[NAMES[k]]:4.0f} % of its tightest axis-aligned 3D box")

🔍 Typically only a third to a half of a 3D bounding box is molecule; for an elongated proteasome lying diagonally even
less. Boxes are fine for *finding* objects; masks are needed for *masking* them (alignment, averaging, measuring).

### 📄 Paper connections — pointers

- **COS-Net.** Zhou B, Yu H, Zeng X, Yang X, Zhang J, Xu M. *One-shot learning with attention-guided segmentation in
  cryo-electron tomography.* Front Mol Biosci 7:613347 (2020; published Jan 2021).
  [doi:10.3389/fmolb.2020.613347](https://doi.org/10.3389/fmolb.2020.613347). Siamese encoder–decoder with **DuSE**
  (dual squeeze-and-excitation) attention blocks — "attention-guided" refers to these blocks, not to thresholding a
  CAM — a 3D CRF, and loss BCE (matching) + Dice (segmentation). Simulated data only (32³, 22 classes, 14 training and
  8 unseen test classes). The authors report 2-way 1-shot accuracy 0.939 at SNR 1000 (0.928 without the decoder), > 0.8
  at SNR 0.5, and segmentation DSC > 0.92 at SNR ∞ and > 0.84 at SNR 1000 for all classes. Code: AITom
  `aitom/classify/deep/supervised/cnn/one_shot/`.
- **Faster-RCNN for mitochondria.** Li R, Zeng X, Sigmund SE, Lin R, Zhou B, Liu C, Wang K, Jiang R, Freyberg Z, Lv H,
  Xu M. *Automatic localization and identification of mitochondria in cellular electron cryo-tomography using
  faster-RCNN.* BMC Bioinformatics 20(Suppl 3):132 (2019). [doi:10.1186/s12859-019-2650-7](https://doi.org/10.1186/s12859-019-2650-7).
  A standard Faster-RCNN (ResNet-50 backbone, region proposal network, 7 × 7 RoI pooling) on **2D** images of rat INS-1E
  cells; the decisive step was preprocessing — bilateral filtering *then* histogram equalisation (the reverse order
  amplified noise). The authors report AP ≥ 0.95 at IoU ≤ 0.8 with preprocessing (≈ 0.3 at IoU 0.8 without), and
  per-tomogram F1 0.91–1.0 except one very low-SNR tomogram. (The paper calls its inputs both "2D projection images" and
  "2D slices".) Code: AITom `aitom/segmentation/detection/organelle/frcnn/`.
- **Point-set networks (AttPNet)** — Yang Y, Ma Y, Zhang J, Gao X, Xu M. *AttPNet: Attention-based deep neural network
  for 3D point set analysis.* Sensors 20(19):5455 (2020); code `aitom/segmentation/point_cloud/deep/attpnet/`.
  ⚠️ Content not verified for this course — read the paper before relying on details.
- **Unsupervised saliency (PUB-SalNet)** — Chen *et al.* *PUB-SalNet: A pre-trained unsupervised self-aware
  backpropagation network for biomedical salient segmentation.* Algorithms 13(5):126 (2020); code
  `aitom/segmentation/saliency/deep/unsupervised/pub_salnet/`. ⚠️ Content not verified for this course.
- The classical (non-deep) saliency method of Zhou *et al.* 2018 (supervoxels + robust PCA) is covered in Module 11;
  foundation-model segmentation (CryoSAM) and topology in Module 32.

---
## ✅ Checkpoint

1. Write the Respond-CAM weight for channel $l$ and explain in one sentence how it differs from Grad-CAM's.
   <details><summary>Answer</summary>α_l = Σ A·∂y/∂A / (Σ A + ε): the gradient averaged with the activations as weights, i.e. only where the channel is active; Grad-CAM averages the gradient uniformly over all voxels.</details>
2. Under which condition do Respond-CAM's voxels sum *exactly* to the class score, and what is left over?
   <details><summary>Answer</summary>When the CAM layer feeds a single fully connected (linear) layer; the sum equals y − b, i.e. the bias of that class is left over. (More generally: when everything after the CAM layer is positively homogeneous — linear layers without hidden biases and ReLUs.)</details>
3. A classifier is 95 % accurate on your test set, but its maps for correct predictions highlight a corner of the box.
   What do you suspect and how would you test it?
   <details><summary>Answer</summary>A shortcut (e.g. a bead, a box-edge artefact, a tomogram-specific feature). Test on data where the suspected cue is removed or added to other classes, or split by tomogram, and see whether accuracy collapses.</details>
4. A predicted mask has IoU 0.5 with the truth. What is its Dice?
   <details><summary>Answer</summary>Dice = 2·IoU/(1 + IoU) = 1/1.5 ≈ 0.67.</details>
5. In the multi-task network, what supervision does the recovery head receive, and why can it still help?
   <details><summary>Answer</summary>A pseudo ground truth produced without human labels (an unsupervised model's output in DSM-Net; a smoothed input here). It adds a dense, label-free training signal that shapes the shared encoder.</details>
6. Name two pieces of prior knowledge our weakly supervised pipeline used besides the image labels.
   <details><summary>Answer</summary>The molecule's mass (to fix the pseudo-mask size) and the assumption that same-class boxes share a shape (cross-image consensus); also that the target is centred, implicitly, through the training data.</details>

## 🧠 Questions & discussion

1. Respond-CAM maps can be negative. What biological meaning could "evidence against capped proteasome" have in a box
   that contains a 20S proteasome?
2. Our CAMs live on a 6³ grid of 6-nm cells. How would you get sharper maps — and what would you lose?
3. The gold-bead shortcut was easy to see. Design a subtler shortcut that might occur in real cryo-ET data and an
   experiment that would expose it.
4. When might multi-task learning *hurt* one of the tasks? (Think about conflicting gradients and loss weights.)
5. CIVA-Net excludes foreground–foreground voxel pairs from its affinity loss "because structures can have holes".
   Which molecules in our zoo have holes or channels, and why would that matter?

## 🚀 Bonus — going further

- **Respond-CAM on a deeper network.** Compute Respond-CAM at the 12³ layer (`f2`) instead of the 6³ layer; the
  sum-to-score property no longer holds exactly — measure by how much.
- **Affinity refinement.** Implement a simple version of CIVA-Net's step 4: train a network to predict whether two
  voxels within 2 voxels of each other share a pseudo-label, and use it to clean the pseudo-masks.
- **One-shot.** Build a tiny Siamese version of `SubtomoNet` (shared encoder; L1 distance of pooled features → "same
  class?") and test it on a class it never saw (e.g. 4CR2).
- **Official code.** Run AITom's Respond-CAM demo (`aitom/segmentation/respond_cam/demo`).
- **Deep dives.** [`TomogramSegmentation`](../../TomogramSegmentation/) and the image-segmentation track
  [`ImageSegmentation`](../../../../sci_img/tutorial/SpatialPerceptionAnalysis/ImageSegmentation/).

## 📝 Lab note

```text
Module 17 — date:
Input: crowded 24³ subtomograms (15 Å voxels, SNR 0.1, ±60° wedge) of 4 zoo targets + neighbours; mass-based masks
Output: Grad-CAM/Respond-CAM maps; sum-to-score errors; shortcut test; segmentation Dice/IoU; multi-task and weak-label results
Method: autograd CAMs; encoder–decoder (CE + soft Dice); shared encoder with 1:10:1 losses; CAM + consensus pseudo-masks
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Zhao 2018 (Respond-CAM); Liu 2018 ICIP & BMVC; Zhu 2021 (CIVA-Net); Zeng 2018; Zhou 2020; Li 2019
```

## 📚 References

1. Zhao G, Zhou B, Wang K, Jiang R, Xu M. Respond-CAM: Analyzing deep models for 3D imaging data by visualizations. MICCAI 2018, LNCS, pp. 485–492. arXiv:1806.00102
2. Liu C, Zeng X, Lin R, Liang X, Freyberg Z, Xing E, Xu M. Deep learning based supervised semantic segmentation of electron cryo-subtomograms. IEEE ICIP 2018, pp. 1578–1582. arXiv:1802.04087
3. Liu C, Zeng X, Wang K, Guo Q, Xu M. Multi-task learning for macromolecule classification, segmentation and coarse structural recovery in cryo-tomography. BMVC 2018. arXiv:1805.06332
4. Zhu X, Chen J, Zeng X, Liang J, Li C, Liu S, Behpour S, Xu M. Weakly supervised 3D semantic segmentation using cross-image consensus and inter-voxel affinity relations. ICCV 2021, pp. 2834–2844. doi:10.1109/ICCV48922.2021.00283
5. Zeng X, Leung MR, Zeev-Ben-Mordehai T, Xu M. A convolutional autoencoder approach for mining features in cellular electron cryo-tomograms and weakly supervised coarse segmentation. *J Struct Biol* 202(2):150–160 (2018). doi:10.1016/j.jsb.2017.12.015
6. Zhou B, Yu H, Zeng X, Yang X, Zhang J, Xu M. One-shot learning with attention-guided segmentation in cryo-electron tomography. *Front Mol Biosci* 7:613347 (2020). doi:10.3389/fmolb.2020.613347
7. Li R, Zeng X, Sigmund SE, *et al.*, Xu M. Automatic localization and identification of mitochondria in cellular electron cryo-tomography using faster-RCNN. *BMC Bioinformatics* 20(Suppl 3):132 (2019). doi:10.1186/s12859-019-2650-7
8. Yang Y, Ma Y, Zhang J, Gao X, Xu M. AttPNet: Attention-based deep neural network for 3D point set analysis. *Sensors* 20(19):5455 (2020). *(content not verified)*
9. Chen *et al.* PUB-SalNet: A pre-trained unsupervised self-aware backpropagation network for biomedical salient segmentation. *Algorithms* 13(5):126 (2020). *(content not verified)*
10. Selvaraju RR, Cogswell M, Das A, Vedantam R, Parikh D, Batra D. Grad-CAM: Visual explanations from deep networks via gradient-based localization. ICCV 2017. doi:10.1109/ICCV.2017.74
11. Zhou B, Khosla A, Lapedriza A, Oliva A, Torralba A. Learning deep features for discriminative localization (CAM). CVPR 2016. doi:10.1109/CVPR.2016.319
12. Ronneberger O, Fischer P, Brox T. U-Net: Convolutional networks for biomedical image segmentation. MICCAI 2015. doi:10.1007/978-3-319-24574-4_28
13. Milletari F, Navab N, Ahmadi SA. V-Net: Fully convolutional neural networks for volumetric medical image segmentation (soft Dice loss). 3DV 2016. doi:10.1109/3DV.2016.79

## ➡️ Next

**[18 · Teaching a network to annotate a cell: CNN segmentation of cellular tomograms](18_cnn_annotation_cellular_tomograms.ipynb)**
— from segmenting one subtomogram to annotating membranes, filaments and ribosomes through a whole tomogram, with
networks trained on ten labelled tiles each.
Previous: [16 · 3D CNN classification](16_3d_cnn_classification.ipynb). Course overview: [`README.md`](../README.md).